# Evaluate UniDepthLSS

Evaluate vehicle IoU at the training resolution, `294 × 518`, both without a visibility filter and with the NuScenes visibility filter (`visibility >= 2`). Both metrics use the same predictions in one pass.

In [1]:
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DATA_ROOT = Path("/data/adeel/data/nuscenes")
CHECKPOINT = Path("/home/adeel/UniDepth_BEV/27_May-VGGTBeV/checkpoints/best_model.pt")
UNIDEPTH_ROOT = Path("/home/adeel/UniDepth")
sys.path.insert(0, str(UNIDEPTH_ROOT))

from dataset_nuscenes import NuScenesBEVDataset
from model import UniDepthLSS
from train_utils import BinaryIoU

IMAGE_SIZE = (294, 518)
BEV_SIZE = 128
BEV_RESOLUTION = 0.5
FEATURE_CHANNELS = 128
MIN_VISIBILITY = 2
THRESHOLD = 0.5
BATCH_SIZE = 1
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for path in (DATA_ROOT, CHECKPOINT, UNIDEPTH_ROOT):
    if not path.exists():
        raise FileNotFoundError(path)

In [2]:
dataset = NuScenesBEVDataset(
    dataroot=str(DATA_ROOT), version="v1.0-trainval", split="val",
    img_size=IMAGE_SIZE, bev_size=BEV_SIZE, bev_res=BEV_RESOLUTION,
    augment=False, return_visibility=True,
)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

model = UniDepthLSS(
    img_height=IMAGE_SIZE[0], img_width=IMAGE_SIZE[1],
    num_classes=1, feature_channels=FEATURE_CHANNELS,
).to(DEVICE)
checkpoint = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

/home/adeel/anaconda3/envs/VGGT/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/adeel/UniDepth/unidepth/utils/chamfer_distance.py:9: UserWarning: !! To run evaluation you need KNN. Please compile KNN: `cd unidepth/ops/knn with && bash compile.sh`.
  warnings.warn(


xFormers not available


xFormers not available


Cannot import NystromAttention, you can not run original UniDepth. UniDepthV2 is available.


Not loading pretrained weights for backbone


UniDepthLSS(
  (projector): _LiftSplatProjector(
    (backbone): UniDepthV2(
      (pixel_encoder): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-23): 24 x NestedTensorBlock(
            (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=1024, out_features=3072, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=1024, out_features=1024, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=1024, out_features=4096, bias=True)
              (act)

In [3]:
without_visibility_filter = BinaryIoU(threshold=THRESHOLD)
with_visibility_filter = BinaryIoU(threshold=THRESHOLD)

with torch.inference_mode():
    for images, intrinsics, extrinsics, target, visibility in tqdm(loader):
        images = images.to(DEVICE, non_blocking=True)
        intrinsics = intrinsics.to(DEVICE, non_blocking=True)
        extrinsics = extrinsics.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        visibility = visibility.to(DEVICE, non_blocking=True)
        with torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"
        ):
            logits = model(images, intrinsics, extrinsics)
        without_visibility_filter.update(logits, target)
        with_visibility_filter.update(
            logits, target, valid_mask=visibility >= MIN_VISIBILITY
        )

print(f"Without Visibility Filter — Vehicle IoU: {without_visibility_filter.compute():.4f}")
print(f"With Visibility Filter    — Vehicle IoU: {with_visibility_filter.compute():.4f}")

  0%|          | 0/6019 [00:00<?, ?it/s]

Without Visibility Filter — Vehicle IoU: 0.4551
With Visibility Filter    — Vehicle IoU: 0.4849
